<a href="https://colab.research.google.com/github/eyramg/ds2002-fa26/blob/main/2026_10_09_%E2%80%94_Public_API_to_Clean_Table_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [34]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat,
        'longitude': lon,
        'start_date': start,
        'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }

    for attempt in range(tries):
        try:
            r = requests.get(BASE, params=params, timeout=15)

            if r.status_code == 200:
                return pd.DataFrame(r.json()['daily'])

            if r.status_code not in RETRY_STATUSES:
                r.raise_for_status()

            r.raise_for_status()

        except requests.RequestException as e:
            status = getattr(e.response, 'status_code', None)

            if status is not None and status not in RETRY_STATUSES:
                raise

            if attempt == tries - 1:
                raise

            time.sleep(2 ** attempt)

    raise RuntimeError("Weather request failed")

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [35]:
used_fallback = False

try:
    weather_data = get_weather('2026-09-05', '2026-10-03')
    wx = weather_data[
        weather_data['time'].isin(['2026-09-05', '2026-10-03'])
    ].copy()

    if len(wx) != 2:
        raise ValueError("Missing weather data for one or both dates")

except (requests.RequestException, ValueError, KeyError) as e:
    print("API failed:", e)
    used_fallback = True
    wx = None

print("Used fallback:", used_fallback)

Used fallback: False


### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [36]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-03'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})

if used_fallback:
    wx = FALLBACK.copy()
    wx['source'] = 'fallback sample'
else:
    wx['source'] = 'open-meteo'

print(wx)

          time  temperature_2m_max  precipitation_sum      source
0   2026-09-05                29.4                2.5  open-meteo
28  2026-10-03                22.0               14.7  open-meteo


### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [37]:
wx = wx.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm'
})

wx['date'] = pd.to_datetime(wx['date'])

wx = wx[['date', 'temp_max', 'precip_mm', 'source']]

print(wx)

         date  temp_max  precip_mm      source
0  2026-09-05      29.4        2.5  open-meteo
28 2026-10-03      22.0       14.7  open-meteo


### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [38]:
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'

print('weather table is safe to join.')

weather table is safe to join.


### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [39]:
wettest = wx.loc[wx['precip_mm'].idxmax()]
driest = wx.loc[wx['precip_mm'].idxmin()]

difference = wettest['precip_mm'] - driest['precip_mm']

print(
    f"{wettest['date'].strftime('%B %d, %Y')} was wetter than "
    f"{driest['date'].strftime('%B %d, %Y')} by "
    f"{difference:.1f} mm of precipitation."
)

October 03, 2026 was wetter than September 05, 2026 by 12.2 mm of precipitation.


### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [40]:
from datetime import datetime, timezone

print("API endpoint:", BASE)
print("Coordinates: 38.03, -78.51")
print("Date range: 2026-09-05 to 2026-10-03")
print("Retrieved:", datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M UTC'))
print("Data source:", "fallback sample" if used_fallback else "open-meteo")

API endpoint: https://archive-api.open-meteo.com/v1/archive
Coordinates: 38.03, -78.51
Date range: 2026-09-05 to 2026-10-03
Retrieved: 2026-10-12 03:33 UTC
Data source: open-meteo


### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

If `get_weather` returned an empty DataFrame instead of raising an error, the weather data would be missing when merged with the 200 days of sales. Depending on the type of merge, the chart might show missing rainfall values or leave out sales records entirely. This could lead to misleading conclusions about the relationship between rain and revenue. The problem might go unnoticed if nobody checks for missing data or validates the results before creating the chart.

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [41]:
cities = {
    'Charlottesville': (38.03, -78.51),
    'Orlando': (28.54, -81.38)
}

all_weather = []

for city, (lat, lon) in cities.items():
    try:
        data = get_weather(
            '2026-09-05',
            '2026-10-03',
            lat=lat,
            lon=lon
        )

        data = data[
            data['time'].isin(['2026-09-05', '2026-10-03'])
        ].copy()

        data['city'] = city
        data['source'] = 'open-meteo'

        all_weather.append(data)

    except requests.RequestException as e:
        print(f"Could not retrieve weather for {city}: {e}")

if all_weather:
    combined = pd.concat(all_weather, ignore_index=True)

    combined = combined.rename(columns={
        'time': 'date',
        'temperature_2m_max': 'temp_max',
        'precipitation_sum': 'precip_mm'
    })

    combined['date'] = pd.to_datetime(combined['date'])

    combined = combined[
        ['city', 'date', 'temp_max', 'precip_mm', 'source']
    ]

    print(combined)
else:
    print("No weather data retrieved.")

              city       date  temp_max  precip_mm      source
0  Charlottesville 2026-09-05      29.4        2.5  open-meteo
1  Charlottesville 2026-10-03      22.0       14.7  open-meteo
2          Orlando 2026-09-05      32.1        3.9  open-meteo
3          Orlando 2026-10-03      30.4       14.5  open-meteo
